# Prediksi Putus Kuliah Mahasiswa (Putus Kuliah vs Lulus)
**Jenis masalah:** Klasifikasi (target: Putus Kuliah atau Lulus) | **Algoritma:** Random Forest | **Dataset:** UCI *Predict Students' Dropout and Academic Success* (SDGs 10)

**Alur:** Muat Data → Heatmap Korelasi (EDA) → Pilih Fitur → Bagi Data 80:20 → Klasifikasi/Regresi → Algoritma → Evaluasi (Tabel dan Confusion Matrix) → Inference

## 1. Muat Data
Dataset aslinya berbahasa Inggris, jadi nama kolom dan label statusnya diterjemahkan ke bahasa Indonesia lewat kamus `terjemah` di bawah.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

URL = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
df = pd.read_csv(URL, sep=";")
df.columns = df.columns.str.strip().str.lower()

# Kamus terjemahan nama kolom (Inggris -> Indonesia)
terjemah = {
    "marital status": "status pernikahan",
    "application mode": "jalur pendaftaran",
    "application order": "urutan pilihan pendaftaran",
    "course": "program studi",
    "daytime/evening attendance": "kuliah siang",
    "previous qualification": "pendidikan sebelumnya",
    "previous qualification (grade)": "nilai pendidikan sebelumnya",
    "nacionality": "kewarganegaraan",
    "mother's qualification": "pendidikan ibu",
    "father's qualification": "pendidikan ayah",
    "mother's occupation": "pekerjaan ibu",
    "father's occupation": "pekerjaan ayah",
    "admission grade": "nilai masuk",
    "displaced": "perantau",
    "educational special needs": "kebutuhan khusus",
    "debtor": "punya tunggakan",
    "tuition fees up to date": "SPP lancar",
    "gender": "jenis kelamin",
    "scholarship holder": "penerima beasiswa",
    "age at enrollment": "usia saat masuk",
    "international": "mahasiswa internasional",
    "unemployment rate": "tingkat pengangguran",
    "inflation rate": "tingkat inflasi",
    "gdp": "PDB",
    "target": "status",
}
# Kolom mata kuliah per semester
istilah = {"credited": "diakui", "enrolled": "diambil", "evaluations": "dievaluasi",
           "approved": "lulus", "grade": "nilai", "without evaluations": "tanpa evaluasi"}
for sem, no in [("1st", 1), ("2nd", 2)]:
    for en, idn in istilah.items():
        terjemah[f"curricular units {sem} sem ({en})"] = f"mata kuliah smt {no} ({idn})"

df = df.rename(columns=terjemah)

print("Jumlah baris dan kolom:", df.shape)
print("Jumlah data kosong    :", df.isnull().sum().sum())
print(df["status"].value_counts())
df.head()

Kelas **Enrolled** (masih kuliah) dibuang karena hasil akhirnya belum ada. Sisanya diubah jadi angka: `1 = Putus Kuliah`, `0 = Lulus`.

Catatan arti kode: `jenis kelamin` 1 = laki-laki, 0 = perempuan. Kolom yang berisi 0/1 seperti `penerima beasiswa`, `SPP lancar`, dan `punya tunggakan` artinya 1 = ya, 0 = tidak.

In [ ]:
df = df[df["status"] != "Enrolled"].copy().reset_index(drop=True)
df["status"] = df["status"].map({"Graduate": "Lulus", "Dropout": "Putus Kuliah"})
df["putus_kuliah"] = (df["status"] == "Putus Kuliah").astype(int)

print("Ukuran data:", df.shape)
print(df["status"].value_counts())

## 2. Heatmap Korelasi (EDA)
Sebelum memilih fitur, kita lihat dulu hubungan antar kolom lewat heatmap korelasi. Agar terbaca, ditampilkan **15 kolom dengan korelasi tertinggi terhadap `putus_kuliah`** (dari seluruh kolom yang ada).

In [ ]:
korelasi = df.drop(columns=["status"]).corr()["putus_kuliah"].drop("putus_kuliah")
urutan = korelasi.abs().sort_values(ascending=False).index.tolist()   # kolom diurutkan dari korelasi terkuat

kandidat = ["putus_kuliah"] + urutan[:15]
plt.figure(figsize=(12, 10))
sns.heatmap(df[kandidat].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Heatmap Korelasi: 15 Kolom Terkuat terhadap Putus Kuliah")
plt.tight_layout()
plt.show()

**Cara baca:** angka mendekati +1 atau -1 berarti hubungan kuat. Nilai positif = makin tinggi, makin besar kemungkinan putus kuliah. Nilai negatif = sebaliknya (misalnya makin banyak mata kuliah lulus, makin kecil kemungkinan putus kuliah). Perhatikan juga fitur yang saling berkorelasi tinggi (misalnya mata kuliah lulus semester 1 dan 2): keduanya membawa informasi yang mirip.

## 3. Memilih Fitur
Berdasarkan heatmap, fitur dipilih dari **10 kolom dengan korelasi absolut tertinggi** terhadap `putus_kuliah`.

In [ ]:
fitur = urutan[:10]

print("Fitur terpilih:")
for f in fitur:
    print(f"  {f:40s} korelasi = {korelasi[f]:+.3f}")

## 4. Bagi Data Latih dan Data Uji (80:20)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[fitur]
y = df["putus_kuliah"]

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Data latih:", X_latih.shape)
print("Data uji  :", X_uji.shape)

## 5. Regresi atau Klasifikasi?
**Klasifikasi.** Targetnya berupa kategori (Putus Kuliah atau Lulus), bukan angka kontinu seperti harga atau suhu. Kalau targetnya angka (misalnya memprediksi IPK), baru pakai regresi.

## 6. Algoritma: Random Forest
Random Forest membuat banyak pohon keputusan, lalu hasil akhirnya diambil dari **voting** semua pohon. Cocok untuk pemula karena stabil, tidak butuh penskalaan data, dan jarang overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_latih, y_latih)
print("Model selesai dilatih.")

## 7. Evaluasi Hasil
Model diuji pada **data uji** yang belum pernah dilihat saat pelatihan.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_prediksi = model.predict(X_uji)

print("Akurasi data latih:", round(accuracy_score(y_latih, model.predict(X_latih)), 3))
print("Akurasi data uji  :", round(accuracy_score(y_uji, y_prediksi), 3))

laporan = pd.DataFrame(classification_report(
    y_uji, y_prediksi, target_names=["Lulus", "Putus Kuliah"], output_dict=True)).T
laporan = laporan.rename(columns={"precision": "presisi", "recall": "recall",
                                  "f1-score": "f1-score", "support": "jumlah data"})
laporan = laporan.rename(index={"accuracy": "akurasi", "macro avg": "rata-rata makro",
                                "weighted avg": "rata-rata berbobot"})

laporan = laporan.round(3).astype(object)
laporan["jumlah data"] = laporan["jumlah data"].astype(float).astype(int)
laporan.loc["akurasi", ["presisi", "recall"]] = ""
laporan.loc["akurasi", "jumlah data"] = len(y_uji)

display(laporan)

**Cara baca:** *presisi* = dari semua yang ditebak Putus Kuliah, berapa persen yang benar. *Recall* = dari semua mahasiswa yang benar-benar putus kuliah, berapa persen yang berhasil terdeteksi.

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_uji, y_prediksi)

# Tabel
tabel_cm = pd.DataFrame(cm,
                        index=["Aktual: Lulus", "Aktual: Putus Kuliah"],
                        columns=["Prediksi: Lulus", "Prediksi: Putus Kuliah"])
display(tabel_cm)

# Heatmap
plt.figure(figsize=(5.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Lulus", "Putus Kuliah"], yticklabels=["Lulus", "Putus Kuliah"])
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

**Cara baca:** diagonal (kiri atas dan kanan bawah) = prediksi benar. Kanan atas = mahasiswa Lulus yang salah ditebak Putus Kuliah. Kiri bawah = mahasiswa Putus Kuliah yang **tidak terdeteksi** (kesalahan yang paling merugikan untuk deteksi dini).

Fitur yang paling berpengaruh menurut model:

In [ ]:
penting = pd.Series(model.feature_importances_, index=fitur).sort_values()
penting.plot(kind="barh", figsize=(8, 4), color="steelblue")
plt.title("Tingkat Kepentingan Fitur")
plt.xlabel("Tingkat kepentingan")
plt.tight_layout()
plt.show()

## 8. Inference
Inference = memakai model yang sudah dilatih untuk memprediksi **data baru**. Di sini kamu bisa **mengetik sendiri nilai tiap fitur**, lalu model menebak apakah mahasiswa itu Putus Kuliah atau Lulus.

Cara pakai: jalankan sel di bawah, isi nilai tiap fitur, lalu tekan Enter. Kalau langsung Enter tanpa mengetik, dipakai nilai default (median untuk angka biasa, nilai paling sering muncul untuk kategori).

Input dicek otomatis supaya tidak salah:
- **Fitur 0/1** (misalnya `penerima beasiswa`): hanya boleh 0 atau 1.
- **Fitur kategori berupa kode** (misalnya `jalur pendaftaran`): daftar kode yang valid ditampilkan lebih dulu, dan kode di luar daftar ditolak.
- **Fitur angka biasa** (misalnya `usia saat masuk`): harus berada di rentang min sampai maks data latih.

In [ ]:
# Kolom yang isinya KODE kategori (angkanya hanya label, bukan jumlah/urutan)
kolom_kategori = ["status pernikahan", "jalur pendaftaran", "program studi", "pendidikan sebelumnya",
                  "kewarganegaraan", "pendidikan ibu", "pendidikan ayah", "pekerjaan ibu", "pekerjaan ayah"]

# Arti kode (sesuai tabel variabel dokumentasi dataset UCI); cocokkan lagi dengan dokumentasi sebelum dipresentasikan
arti_kode = {
    "jalur pendaftaran": {
        1: "Tahap 1 - kontingen umum", 2: "Peraturan No. 612/93", 5: "Tahap 1 - kontingen khusus (Kep. Azores)",
        7: "Pemegang gelar pendidikan tinggi lain", 10: "Peraturan No. 854-B/99", 15: "Mahasiswa internasional (S1)",
        16: "Tahap 1 - kontingen khusus (Kep. Madeira)", 17: "Tahap 2 - kontingen umum", 18: "Tahap 3 - kontingen umum",
        26: "Peraturan No. 533-A/99 b2 (rencana studi berbeda)", 27: "Peraturan No. 533-A/99 b3 (institusi lain)",
        39: "Usia di atas 23 tahun", 42: "Pindahan (transfer)", 43: "Pindah program studi",
        44: "Pemegang diploma spesialisasi teknologi", 51: "Pindah institusi/program studi",
        53: "Pemegang diploma siklus pendek", 57: "Pindah institusi/program studi (internasional)",
    },
    "status pernikahan": {
        1: "Lajang", 2: "Menikah", 3: "Duda/janda", 4: "Cerai", 5: "Hidup bersama tanpa menikah", 6: "Pisah secara hukum",
    },
}

def prediksi(data):
    prob = model.predict_proba(data)[0, 1]
    label = "Putus Kuliah" if prob >= 0.5 else "Lulus"
    return label, round(prob * 100, 1)

def input_mahasiswa():
    """Meminta pengguna mengisi nilai tiap fitur lewat keyboard, dengan pengecekan input."""
    nilai = {}
    print("Masukkan nilai fitur (Enter = pakai nilai default).")
    for f in fitur:
        valid = sorted(X_latih[f].unique())
        mn, mx = X_latih[f].min(), X_latih[f].max()

        if set(valid) <= {0, 1}:                      # fitur 0/1
            jenis, default = "pilihan", X_latih[f].mode()[0]
            petunjuk = "1 = laki-laki, 0 = perempuan" if f == "jenis kelamin" else "1 = ya, 0 = tidak"
            tanya = f"{f}  [{petunjuk}, default {default:g}]: "
        elif f in kolom_kategori:                     # kode kategori
            jenis, default = "pilihan", X_latih[f].mode()[0]
            print(f"\nPilihan kode untuk '{f}':")
            for k in valid:
                arti = arti_kode.get(f, {}).get(int(k), "")
                print(f"   {int(k)}" + (f" = {arti}" if arti else ""))
            tanya = f"{f}  [pilih salah satu kode di atas, default {default:g}]: "
        else:                                         # angka biasa
            jenis, default = "rentang", X_latih[f].median()
            tanya = f"{f}  [min {mn:g} - maks {mx:g}, default {default:g}]: "

        while True:
            jawab = input(tanya).strip()
            if jawab == "":
                nilai[f] = default
                break
            try:
                angka = float(jawab)
            except ValueError:
                print("   Harus berupa angka, coba lagi.")
                continue
            if jenis == "pilihan" and angka not in valid:
                print("   Kode tidak tersedia, pilih salah satu dari daftar.")
                continue
            if jenis == "rentang" and not (mn <= angka <= mx):
                print(f"   Di luar rentang data latih ({mn:g} - {mx:g}), coba lagi.")
                continue
            nilai[f] = angka
            break
    return pd.DataFrame([nilai])[fitur]

mhs = input_mahasiswa()
label, prob = prediksi(mhs)

print("\n=== HASIL PREDIKSI ===")
print("Prediksi                 :", label)
print(f"Probabilitas putus kuliah: {prob}%")


Masukkan nilai fitur (Enter = pakai nilai default).


In [ ]:
# Contoh cepat tanpa mengetik: mahasiswa berisiko (ubah angkanya sesuka kamu)
kasus = X_latih.median().to_frame().T
for kolom, nilai in {
    "SPP lancar": 0,
    "penerima beasiswa": 0,
    "mata kuliah smt 1 (lulus)": 0,
    "mata kuliah smt 2 (lulus)": 0,
}.items():
    if kolom in kasus.columns:
        kasus[kolom] = nilai

label, prob = prediksi(kasus)
print(f"Contoh mahasiswa berisiko -> prediksi: {label} (probabilitas putus kuliah {prob}%)")

In [ ]:
# Prediksi 5 mahasiswa dari data uji sekaligus, dibandingkan dengan kenyataan
nama = {0: "Lulus", 1: "Putus Kuliah"}
hasil = pd.DataFrame({
    "Aktual": y_uji.head(5).map(nama).values,
    "Prediksi": pd.Series(model.predict(X_uji.head(5))).map(nama).values,
    "Prob. putus kuliah (%)": (model.predict_proba(X_uji.head(5))[:, 1] * 100).round(1),
})
hasil

## 9. Kesimpulan
*(Isi setelah semua sel dijalankan, pakai angka dari output.)*
1. Jenis masalah: klasifikasi, algoritma Random Forest, 10 fitur dipilih dari korelasi tertinggi.
2. Akurasi data uji = ... (bandingkan dengan akurasi data latih; selisih besar tanda overfitting).
3. Dari confusion matrix: ... mahasiswa putus kuliah terdeteksi, ... tidak terdeteksi.
4. Fitur paling berpengaruh: ...

**Keterbatasan**
- Kelas Enrolled dibuang karena hasil akhirnya belum diketahui, sehingga model hanya berlaku untuk mahasiswa yang statusnya sudah final.
- Heatmap dan pemilihan fitur memakai seluruh data sebelum dibagi 80:20 (sesuai alur tugas); idealnya memakai data latih saja.